# Notebook 8 — a mining task with a real evaluation

Notebook 7 produced a mixed result. The higher-order triadic profile is recoverable from a
stream to within $2\%$, and the predictor cuts the instances needed for $5\%$ error from
$5000$ to $2000$; but the profile does not separate dataset domains, since leave-one-out
accuracy is $0.30$ against a chance level of $0.13$, and the drug hypergraphs are spread
across the whole range. Ten graphs were never going to support a classification claim.

This notebook moves the question inside a single graph, where the sample size is thousands of
vertices rather than ten datasets.

### The task

For a vertex $v$, let $c(v)$ be the number of cross-hyperedge triangles containing it: the
extent to which $v$ sits at the centre of genuine higher-order closure rather than at the
centre of a single large group. It is obtainable exactly from the per-edge counts, since each
such triangle meets $v$ in exactly two $v$-incident edges,
\[
c(v) \;=\; \tfrac{1}{2}\sum_{e \ni v} t(e),
\]
but computing $t(e)$ exactly costs a neighbourhood intersection per edge, which is precisely
what a streaming setting rules out.

The question is therefore the standard one for a mining primitive: **can a stream identify the
top of this ranking, and at what cost?** The evaluation is precision at $K$ against the exact
ranking, together with the rank correlation over all copy-bearing vertices.

## 0. Library and data

In [ ]:
"""Core graph primitives.

Degeneracy by peeling, exact oracle-width (pseudoarboricity) by binary search over
max-flow feasibility, and the structural bracket

    ceil(kappa_H / 2)  <=  alpha_H  <=  kappa_H  <=  kappa(G)

of Lemma 1 in the paper.
"""
import time
from collections import defaultdict

import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import maximum_flow

__all__ = ['bits_to_idx', 'popcount', 'relabel', 'canon', 'build_adj',
           'degeneracy', 'degeneracy_arrays', 'orientation_feasible', 'oracle_width',
           'bracket']


# ----------------------------------------------------------------- bit helpers

def bits_to_idx(x, nbytes):
    """Indices of the set bits of a Python int."""
    if x == 0:
        return np.empty(0, dtype=np.int32)
    b = np.frombuffer(x.to_bytes(nbytes, 'little'), dtype=np.uint8)
    return np.flatnonzero(np.unpackbits(b, bitorder='little')).astype(np.int32)


def popcount(x):
    try:
        return x.bit_count()
    except AttributeError:              # Python < 3.10
        return bin(x).count('1')


# ------------------------------------------------------------- edge-list helpers

def relabel(edges):
    """Map arbitrary hashable node labels to consecutive integers."""
    ids, out = {}, []
    for u, v in edges:
        for x in (u, v):
            if x not in ids:
                ids[x] = len(ids)
        out.append((ids[u], ids[v]))
    return out, ids


def canon(edges):
    """De-duplicate, drop self-loops, relabel to ints, return sorted (u, v) with u < v."""
    edges, _ = relabel(edges)
    s = set()
    for u, v in edges:
        if u == v:
            continue
        s.add((u, v) if u < v else (v, u))
    return sorted(s)


def build_adj(edges):
    adj = defaultdict(set)
    for u, v in edges:
        if u == v:
            continue
        adj[u].add(v)
        adj[v].add(u)
    return adj


# ------------------------------------------------------------------- degeneracy

def degeneracy(adj):
    """Exact degeneracy of an adjacency dict, by peeling. Returns (kappa, core numbers)."""
    if not adj:
        return 0, {}
    deg = {v: len(adj[v]) for v in adj}
    maxdeg = max(deg.values())
    buckets = [set() for _ in range(maxdeg + 1)]
    for v, d in deg.items():
        buckets[d].add(v)
    core, k, removed = {}, 0, set()
    i = 0
    for _ in range(len(deg)):
        while i <= maxdeg and not buckets[i]:
            i += 1
        if i > maxdeg:
            break
        v = buckets[i].pop()
        k = max(k, i)
        core[v] = k
        removed.add(v)
        for w in adj[v]:
            if w in removed:
                continue
            d = deg[w]
            buckets[d].discard(w)
            deg[w] = d - 1
            buckets[d - 1].add(w)
            if d - 1 < i:
                i = d - 1
    return k, core


def degeneracy_arrays(eu, ev):
    """Degeneracy of the graph given by parallel edge arrays."""
    if len(eu) == 0:
        return 0, 0
    nodes = np.unique(np.concatenate([eu, ev]))
    remap = {int(v): i for i, v in enumerate(nodes)}
    n = len(nodes)
    adj = [[] for _ in range(n)]
    for a, b in zip(eu, ev):
        a, b = remap[int(a)], remap[int(b)]
        adj[a].append(b)
        adj[b].append(a)
    deg = np.array([len(a) for a in adj])
    maxd = int(deg.max())
    buckets = [set() for _ in range(maxd + 1)]
    for v in range(n):
        buckets[deg[v]].add(v)
    removed = np.zeros(n, dtype=bool)
    k, i = 0, 0
    for _ in range(n):
        while i <= maxd and not buckets[i]:
            i += 1
        if i > maxd:
            break
        v = buckets[i].pop()
        k = max(k, i)
        removed[v] = True
        for w in adj[v]:
            if removed[w]:
                continue
            d = int(deg[w])
            buckets[d].discard(w)
            deg[w] = d - 1
            buckets[d - 1].add(w)
            if d - 1 < i:
                i = d - 1
    return k, n


# ------------------------------------------------- oracle-width (pseudoarboricity)

def orientation_feasible(eu, ev, k):
    """Is there an orientation of these edges with maximum out-degree at most k?

    Max-flow feasibility (Hakimi; Frank and Gyarfas):
    s -> edge (cap 1); edge -> each endpoint (cap 1); vertex -> t (cap k).
    Feasible iff the flow saturates all m edges.
    """
    m = len(eu)
    if m == 0:
        return True
    nodes = np.unique(np.concatenate([eu, ev]))
    pos = {int(v): i for i, v in enumerate(nodes)}
    n = len(nodes)
    S, E0, V0 = 0, 1, 1 + m
    T = 1 + m + n
    rows = np.empty(3 * m + n, dtype=np.int64)
    cols = np.empty(3 * m + n, dtype=np.int64)
    data = np.empty(3 * m + n, dtype=np.int32)
    idx = np.arange(m)
    rows[:m] = S
    cols[:m] = E0 + idx
    data[:m] = 1
    rows[m:2 * m] = E0 + idx
    cols[m:2 * m] = V0 + np.array([pos[int(x)] for x in eu])
    data[m:2 * m] = 1
    rows[2 * m:3 * m] = E0 + idx
    cols[2 * m:3 * m] = V0 + np.array([pos[int(x)] for x in ev])
    data[2 * m:3 * m] = 1
    rows[3 * m:] = V0 + np.arange(n)
    cols[3 * m:] = T
    data[3 * m:] = int(k)
    g = csr_matrix((data, (rows, cols)), shape=(T + 1, T + 1))
    return int(maximum_flow(g, S, T).flow_value) == m


def oracle_width(eu, ev, time_budget=None):
    """Exact pseudoarboricity of the edge set; None if the time budget runs out."""
    if len(eu) == 0:
        return 0
    hi, _ = degeneracy_arrays(eu, ev)
    hi = max(hi, 1)
    lo = 1
    t0 = time.time()
    while lo < hi:
        if time_budget is not None and time.time() - t0 > time_budget:
            return None
        mid = (lo + hi) // 2
        if orientation_feasible(eu, ev, mid):
            hi = mid
        else:
            lo = mid + 1
    return lo


def bracket(kappa_copy):
    """The bracket of Lemma 1: (lower bound, upper bound) on alpha_H."""
    return int(np.ceil(kappa_copy / 2)), int(kappa_copy)


"""Projection of a hypergraph, with exact cross-hyperedge-triangle structure.

A triple {u, v, w} is a *cross-hyperedge triangle* when it is a triangle in the projection
and no single hyperedge contains all three vertices.  Per-edge counts come from the closed
form

    t({u,v}) = |N(u) & N(v)|  -  |union of hyperedges containing both u and v, minus {u,v}|

which avoids triangle enumeration; only the list of triples used to sum per-copy path
probabilities is (optionally) subsampled, and every predictor is scored on the same sample,
so the reported ratios are unaffected.
"""
import numpy as np


__all__ = ['CrossHG', 'diagnostic_crossK3']


class CrossHG:
    """Projected graph of a hypergraph + exact cross-triangle structure.

    Cross triangle: {u,v,w} pairwise adjacent in the projection, with no single
    hyperedge containing all three.  Canonical discovery follows the Fichtenberger-Peng
    path used by PredCount: order the three vertices by (degree, id) as x < y < z,
    base edge {x,y}, pivot x, closing vertex z.
    """

    def __init__(self, hyperedges, max_size=25, max_triples=4_000_000, seed=0, verbose=True):
        hs = [tuple(sorted(set(h))) for h in hyperedges]
        hs = [h for h in hs if 2 <= len(h) <= max_size]
        hs = list(dict.fromkeys(hs))
        verts = sorted({v for h in hs for v in h})
        idx = {v: i for i, v in enumerate(verts)}
        self.n = n = len(verts)
        self.nbytes = (n + 7) // 8
        self.hs = [tuple(idx[v] for v in h) for h in hs]
        self.hbits = [sum(1 << a for a in h) for h in self.hs]
        self.hsize = np.array([len(h) for h in self.hs], dtype=np.int32)

        memb = [[] for _ in range(n)]
        adj = [0] * n
        for hid, h in enumerate(self.hs):
            bits = self.hbits[hid]
            for a in h:
                memb[a].append(hid)
                adj[a] |= bits & ~(1 << a)
        self.memb = [set(m) for m in memb]
        self.adj = adj

        # ---- edges, CSR adjacency, edge ids
        nbr = [bits_to_idx(adj[u], self.nbytes) for u in range(n)]
        self.deg = np.array([len(x) for x in nbr], dtype=np.int64)
        self.indptr = np.zeros(n + 1, dtype=np.int64)
        np.cumsum(self.deg, out=self.indptr[1:])
        self.indices = np.concatenate(nbr) if n else np.empty(0, np.int32)

        eu, ev = [], []
        for u in range(n):
            w = nbr[u][nbr[u] > u]
            eu.append(np.full(len(w), u, dtype=np.int32))
            ev.append(w)
        self.eu = np.concatenate(eu) if n else np.empty(0, np.int32)
        self.ev = np.concatenate(ev) if n else np.empty(0, np.int32)
        self.m = len(self.eu)
        self.eid = {}
        for i in range(self.m):
            self.eid[(int(self.eu[i]), int(self.ev[i]))] = i
        # edge id for every CSR slot
        self.slot_eid = np.empty(len(self.indices), dtype=np.int64)
        for u in range(n):
            a, b = self.indptr[u], self.indptr[u + 1]
            for k in range(a, b):
                v = int(self.indices[k])
                self.slot_eid[k] = self.eid[(u, v)] if u < v else self.eid[(v, u)]

        self._core_numbers()
        self.rank = np.empty(n, dtype=np.int64)
        order = np.lexsort((np.arange(n), self.deg))
        self.rank[order] = np.arange(n)

        self._cross_and_triples(max_triples, seed, verbose)
        self._features()

    # ------------------------------------------------------------------ structure
    def _core_numbers(self):
        n = self.n
        deg = self.deg.copy()
        core = np.zeros(n, dtype=np.int64)
        removed = np.zeros(n, dtype=bool)
        maxd = int(deg.max()) if n else 0
        buckets = [set() for _ in range(maxd + 1)]
        for v in range(n):
            buckets[deg[v]].add(v)
        k, i = 0, 0
        for _ in range(n):
            while i <= maxd and not buckets[i]:
                i += 1
            if i > maxd:
                break
            v = buckets[i].pop()
            k = max(k, i)
            core[v] = k
            removed[v] = True
            for j in range(self.indptr[v], self.indptr[v + 1]):
                w = int(self.indices[j])
                if removed[w]:
                    continue
                d = int(deg[w])
                buckets[d].discard(w)
                deg[w] = d - 1
                buckets[d - 1].add(w)
                if d - 1 < i:
                    i = d - 1
        self.core = core
        self.kappa = int(k)

    def _cross_and_triples(self, max_triples, seed, verbose):
        """Per-edge cross-triangle counts (exact) and the triple arrays (possibly sampled)."""
        nb = self.nbytes
        t = np.zeros(self.m, dtype=np.int64)
        covn = np.zeros(self.m, dtype=np.int64)     # covered common neighbours
        nhy = np.zeros(self.m, dtype=np.int64)      # hyperedges containing the edge
        hsum = np.zeros(self.m, dtype=np.int64)     # sum over those of (|h|-2)
        hmax = np.zeros(self.m, dtype=np.int64)
        covbits = [0] * self.m
        for i in range(self.m):
            u, v = int(self.eu[i]), int(self.ev[i])
            common = self.adj[u] & self.adj[v]
            S = self.memb[u] & self.memb[v]
            cov = 0
            for h in S:
                cov |= self.hbits[h]
            covbits[i] = cov
            nhy[i] = len(S)
            if S:
                sz = self.hsize[list(S)]
                hsum[i] = int((sz - 2).sum())
                hmax[i] = int(sz.max())
                covn[i] = popcount(cov & common)
            t[i] = popcount(common & ~cov)
        self.t = t
        self.covn, self.nhy, self.hsum, self.hmax = covn, nhy, hsum, hmax
        self.n_cross_total = int(t.sum()) // 3

        # triples: each cross triangle is generated once, from its canonical base edge
        target = self.n_cross_total if self.n_cross_total else 1
        p = min(1.0, max_triples / target)
        rng = np.random.default_rng(seed)
        keep = np.ones(self.m, dtype=bool) if p >= 1.0 else (rng.random(self.m) < p)
        self.triple_scale = 1.0 / p
        self.triple_fraction = p

        base, pivot, xz = [], [], []
        rank = self.rank
        for i in np.flatnonzero(keep):
            i = int(i)
            if t[i] == 0:
                continue
            u, v = int(self.eu[i]), int(self.ev[i])
            cross = (self.adj[u] & self.adj[v]) & ~covbits[i]
            z = bits_to_idx(cross, nb)
            hi = max(rank[u], rank[v])
            z = z[rank[z] > hi]
            if not len(z):
                continue
            x, y = (u, v) if rank[u] < rank[v] else (v, u)
            base.append(np.full(len(z), i, dtype=np.int32))
            pivot.append(np.full(len(z), x, dtype=np.int32))
            xz.append(np.array([self.eid[(min(x, int(c)), max(x, int(c)))] for c in z],
                               dtype=np.int32))
        self.base = np.concatenate(base) if base else np.empty(0, np.int32)
        self.pivot = np.concatenate(pivot) if pivot else np.empty(0, np.int32)
        self.xz = np.concatenate(xz) if xz else np.empty(0, np.int32)
        if verbose:
            print(f'    n={self.n} m={self.m} kappa={self.kappa} '
                  f'#cross={self.n_cross_total} triples kept={len(self.base)} '
                  f'({100*self.triple_fraction:.1f}% of edges)')

    def _features(self):
        lg = np.log1p
        du, dv = self.deg[self.eu], self.deg[self.ev]
        cu, cv = self.core[self.eu], self.core[self.ev]
        ku = np.array([len(self.memb[u]) for u in self.eu], dtype=np.int64)
        kv = np.array([len(self.memb[v]) for v in self.ev], dtype=np.int64)
        A = np.column_stack([                       # Array-paper block: degrees + cores
            lg(np.minimum(du, dv)), lg(np.maximum(du, dv)),
            lg(du) + lg(dv), np.abs(lg(du) - lg(dv)),
            lg(np.minimum(cu, cv)), lg(np.maximum(cu, cv)), lg(cu) + lg(cv),
        ])
        B = np.column_stack([                       # cheap higher-order block
            lg(np.minimum(ku, kv)), lg(np.maximum(ku, kv)),
            lg(self.nhy), lg(self.hsum), lg(self.hmax),
        ])
        Cc = np.column_stack([lg(self.covn)])       # covered-neighbour block
        self.F = {'A': A, 'AB': np.hstack([A, B]), 'ABC': np.hstack([A, B, Cc])}
        self.y = np.log1p(self.t.astype(float))

    # ------------------------------------------------------------- success probability
    def _D(self, wp):
        """D_x = sum over neighbours c of (w({x,c})+1), as an array over vertices."""
        seg = wp[self.slot_eid]
        out = np.add.reduceat(seg, self.indptr[:-1])
        return out

    def p_succ(self, w, first_edge_only=False):
        wp = np.asarray(w, dtype=float) + 1.0
        W = wp.sum()
        D = self._D(wp)
        if first_edge_only:
            inner = 1.0 / self.deg[self.pivot]
        else:
            inner = wp[self.xz] / D[self.pivot]
        return float((wp[self.base] * inner).sum()) / W * self.triple_scale

    def p_base(self):
        return self.n_cross_total / (2 * self.m) ** 1.5

    # -------------------------------------------------------------------- predictors
    def w_perfect(self):
        return self.t.astype(float)

    def w_uniform(self):
        return np.zeros(self.m)

    def w_mindeg(self):
        return np.minimum(self.deg[self.eu], self.deg[self.ev]).astype(float)

    def w_permuted(self, w, seed):
        rng = np.random.default_rng(seed)
        return rng.permutation(np.asarray(w, dtype=float))


def diagnostic_crossK3(hg, time_budget=900.0):
    """kappa, kappa_copy, exact alpha_H, and the two ratios of the paper's Table 1."""
    import time as _time
    keep = hg.t > 0
    eu, ev = hg.eu[keep], hg.ev[keep]
    kappa_copy, _ = degeneracy_arrays(eu, ev)
    t0 = _time.time()
    alpha = oracle_width(eu, ev, time_budget=time_budget)
    return dict(
        m=hg.m, m_copy=int(keep.sum()), kappa=hg.kappa, kappa_copy=kappa_copy,
        alpha=alpha,
        alpha_lb=int(np.ceil(kappa_copy / 2)), alpha_ub=kappa_copy,
        alpha_over_kappa=(alpha / hg.kappa) if (alpha and hg.kappa) else None,
        kappa_copy_over_kappa=kappa_copy / hg.kappa if hg.kappa else None,
        n_cross=hg.n_cross_total, secs=round(_time.time() - t0, 1),
    )


"""Predictors: the perfect oracle, the permutation null, the cheap heuristic, and a
transferable ridge model over three feature blocks.

Feature blocks (all standardised per graph before fitting):
  A    endpoint degrees and core numbers      -- the block used for ordinary triangles
  AB   A + cheap higher-order features        -- hyperedge membership of the endpoints
  ABC  AB + the covered-common-neighbour count
"""
import numpy as np

__all__ = ['ridge_fit', 'ridge_predict_weights', 'FEATURE_BLOCKS']

FEATURE_BLOCKS = ['A', 'AB', 'ABC']


def ridge_fit(X, y, lam=1.0):
    mx, sx = X.mean(0), X.std(0) + 1e-12
    Z = (X - mx) / sx
    my, sy = y.mean(), y.std() + 1e-12
    yz = (y - my) / sy
    Z1 = np.hstack([Z, np.ones((len(Z), 1))])
    A = Z1.T @ Z1 + lam * np.eye(Z1.shape[1])
    A[-1, -1] -= lam                       # do not penalise the intercept
    beta = np.linalg.solve(A, Z1.T @ yz)
    return dict(beta=beta, mx=mx, sx=sx, my=my, sy=sy)


def ridge_predict_weights(model, X):
    Z = (X - model['mx']) / model['sx']
    Z1 = np.hstack([Z, np.ones((len(Z), 1))])
    yz = Z1 @ model['beta']
    y = yz * model['sy'] + model['my']
    return np.clip(np.expm1(np.clip(y, 0, 30)), 0, None)




"""Reporting: the localization / prediction split, the permutation control, and a
Monte-Carlo check that the analytic success probabilities are the right ones.

`loc_x`            localized but unweighted sampler, over the prediction-free baseline
`perfect_over_loc` what a perfect predictor adds on top of localization
`permuted_*`       the same weight multiset shuffled across edges (the distributional null)
`structural_share` the fraction of the predictor's gain the shuffle does not reproduce
"""
import numpy as np

__all__ = ['evaluate', 'monte_carlo_check']


def evaluate(hg, n_perm=50, seed=0):
    """Localization, perfect predictor, permutation null, cheap heuristic, first-edge."""
    pb = hg.p_base()
    p_loc = hg.p_succ(hg.w_uniform())
    p_perf = hg.p_succ(hg.w_perfect())
    p_mind = hg.p_succ(hg.w_mindeg())
    p_first = hg.p_succ(hg.w_perfect(), first_edge_only=True)
    perms = np.array([hg.p_succ(hg.w_permuted(hg.w_perfect(), seed + b))
                      for b in range(n_perm)])
    s_perf, s_perm = p_perf / p_loc, perms / p_loc
    gain = s_perf - 1.0
    shares = (s_perf - s_perm) / gain if abs(gain) > 1e-12 else np.full(n_perm, np.nan)
    pval = (1 + (s_perm >= s_perf).sum()) / (1 + n_perm)
    return dict(
        n=hg.n, m=hg.m, kappa=hg.kappa, n_cross=hg.n_cross_total,
        copy_density=hg.n_cross_total / max(hg.m, 1),
        triple_fraction=hg.triple_fraction,
        loc_x=p_loc / pb, perfect_x=p_perf / pb, firstedge_x=p_first / pb,
        perfect_over_loc=s_perf,
        permuted_over_loc_mean=float(s_perm.mean()),
        permuted_over_loc_lo=float(np.percentile(s_perm, 2.5)),
        permuted_over_loc_hi=float(np.percentile(s_perm, 97.5)),
        mindeg_over_loc=p_mind / p_loc,
        mindeg_capture=(p_mind / p_loc - 1) / gain if abs(gain) > 1e-12 else np.nan,
        structural_share=float(np.mean(shares)),
        structural_share_lo=float(np.percentile(shares, 2.5)),
        structural_share_hi=float(np.percentile(shares, 97.5)),
        perm_pvalue=float(pval), n_perm=n_perm,
    )


def monte_carlo_check(hg, w, n_samples=200_000, seed=0):
    """Sample real paths from the weighted sampler; compare hit rate to the analytic p."""
    rng = np.random.default_rng(seed)
    wp = np.asarray(w, dtype=float) + 1.0
    pe = wp / wp.sum()
    D = hg._D(wp)
    picks = rng.choice(hg.m, size=n_samples, p=pe)
    hits = 0
    tset = {}
    for i in picks:
        u, v = int(hg.eu[i]), int(hg.ev[i])
        x, y = (u, v) if hg.rank[u] < hg.rank[v] else (v, u)
        a, b = hg.indptr[x], hg.indptr[x + 1]
        slots = hg.slot_eid[a:b]
        pr = wp[slots] / D[x]
        c = int(hg.indices[a + rng.choice(len(slots), p=pr / pr.sum())])
        if c == u or c == v or hg.rank[c] <= hg.rank[y]:
            continue
        key = (min(x, c), max(x, c))
        if key not in hg.eid or (min(y, c), max(y, c)) not in hg.eid:
            continue
        S = hg.memb[x] & hg.memb[y] & hg.memb[c]
        if not S:
            hits += 1
    return hits / n_samples


"""Synthetic families.

Two of these realise the separation regime (Proposition 4 of the paper) and two are
negative controls that do not.
"""
import numpy as np

__all__ = ['friendship_plus_decoy', 'affine_plane_incidence', 'steiner_plus_butterflies',
           'cliques_plus_cross', 'incidence_edges', 'random_uniform_hypergraph']


def friendship_plus_decoy(s):
    """The separation instance of the PredCount paper: F_s together with K_{d,d}.

    kappa = Theta(sqrt(m)) from the decoy, alpha_{K3} = 2 from the friendship graph.
    """
    d = int(np.ceil(np.sqrt(s)))
    E = []
    for i in range(s):
        E += [('h', ('a', i)), ('h', ('b', i)), (('a', i), ('b', i))]
    for i in range(d):
        for j in range(d):
            E.append((('L', i), ('R', j)))
    return E


def affine_plane_incidence(q):
    """Incidence graph of AG(2, q) for prime q.

    Any two points lie on exactly one line, so the incidence graph is C4-free while its
    degeneracy is Theta(q) = Theta(sqrt(m)): a dense region bearing no copies, from
    geometry rather than construction.
    """
    E = []
    for a in range(q):
        for b in range(q):
            lid = ('L', a, b)
            for x in range(q):
                E.append((('P', x, (a * x + b) % q), lid))
    for c in range(q):
        lid = ('V', c)
        for y in range(q):
            E.append((('P', c, y), lid))
    return E


def steiner_plus_butterflies(q, n_gadgets, r=3):
    """AG(2, q) as a copy-free dense core, plus shallow butterfly gadgets.

    C4-freeness alone is vacuous for counting, so pairs co-occurring in two hyperedges are
    added; they are vertex-disjoint, so the copy-bearing subgraph has constant
    pseudoarboricity.
    """
    E = affine_plane_incidence(q)
    for g in range(n_gadgets):
        a, b = ('G', g, 0), ('G', g, 1)
        for t in range(2):
            h = ('H', g, t)
            E.append((a, h))
            E.append((b, h))
            for j in range(r - 2):
                E.append((('F', g, t, j), h))
    return E


def cliques_plus_cross(n_cliques, clique_size, n_gadgets, seed=0):
    """Disjoint large hyperedges (copy-free cliques) plus shallow cross-triangle gadgets.

    A hyperedge of size r projects to K_r, so kappa >= r - 1, but no triple inside it is
    cross: the dense region bears no copies.  This is the hypergraph analogue of the
    friendship-plus-decoy instance, and the family on which localization grows as
    m^{rho(K3) - 1} = m^{0.5}.
    """
    rng = np.random.default_rng(seed)
    H, nxt, cliques = [], 0, []
    for _ in range(n_cliques):
        c = list(range(nxt, nxt + clique_size))
        nxt += clique_size
        H.append(tuple(c))
        cliques.append(c)
    for _ in range(n_gadgets):
        c = cliques[rng.integers(len(cliques))]
        u, v = rng.choice(c, size=2, replace=False)
        w = nxt
        nxt += 1
        H.append((int(u), w))
        H.append((int(v), w))
    return H


def random_uniform_hypergraph(n, m_h, r, seed=0):
    rng = np.random.default_rng(seed)
    return [tuple(rng.choice(n, size=r, replace=False)) for _ in range(m_h)]


def incidence_edges(hyperedges):
    """Bipartite incidence graph of a hyperedge list."""
    E = []
    for i, h in enumerate(hyperedges):
        for v in set(h):
            E.append((('v', v), ('e', i)))
    return E


"""Sample complexity, competing estimators, and a simulation of the real estimator.

Every estimator here is unbiased for #cross, so the cost of reaching a target relative
error is governed by its variance:

    k(eps) = Var[X] / (eps * #cross)^2 ,

which is the number of independent instances the estimator needs.  Reporting all methods
in this single currency makes the comparison exact rather than a horse race between
implementations.
"""
import numpy as np

__all__ = ['second_moment', 'samples_for_error', 'weighted_cost', 'baseline_cost',
           'wedge_cost', 'edge_local_cost', 'heavy_storage_cost', 'simulate_estimator',
           'wedge_weighted_cost', 'wedge_report', 'normalizer_inflation']

EPS = 0.1


def _local_words(hg, mask=None):
    """Words of state a uniformly sampled edge costs a method that counts locally.

    Counting t(e) exactly requires both endpoint neighbourhoods, so the per-sample space
    is deg(u) + deg(v) rather than O(1).  Samplers that make a single closing-vertex draw
    cost one word per instance.
    """
    d = hg.deg[hg.eu] + hg.deg[hg.ev]
    return float(d[mask].mean() if mask is not None else d.mean())


def samples_for_error(var, truth, eps=EPS):
    """Independent instances needed for relative error eps, by Chebyshev."""
    if truth <= 0:
        return np.inf
    return var / (eps * truth) ** 2


# ------------------------------------------------- the weighted / localized estimator

def second_moment(hg, w):
    """E[X^2] for the importance-sampling estimator X = 1 / Pr_w[path], summed over copies.

    Pr_w[C] = (w(base)+1)/W * (w(xz)+1)/D_x, so E[X^2] = sum_C 1/Pr_w[C].
    """
    wp = np.asarray(w, dtype=float) + 1.0
    W = wp.sum()
    D = hg._D(wp)
    pr = (wp[hg.base] / W) * (wp[hg.xz] / D[hg.pivot])
    return float((1.0 / pr).sum()) * hg.triple_scale


def weighted_cost(hg, w, eps=EPS):
    """Variance and sample complexity of the weighted sampler under weights w."""
    truth = hg.n_cross_total
    m2 = second_moment(hg, w)
    var = max(m2 - truth ** 2, 0.0)
    k = samples_for_error(var, truth, eps)
    return dict(second_moment=m2, var=var, samples=k, words=k)


def baseline_cost(hg, eps=EPS):
    """The prediction-free Fichtenberger-Peng sampler: every copy found w.p. (2m)^{-3/2}."""
    truth = hg.n_cross_total
    p = (2 * hg.m) ** 1.5
    var = max(truth * p - truth ** 2, 0.0)
    k = samples_for_error(var, truth, eps)
    return dict(var=var, samples=k, words=k)


# ------------------------------------------------------------------- competing methods

def wedge_cost(hg, eps=EPS):
    """Uniform wedge sampling: draw one of the sum_v C(deg v, 2) wedges, test closure.

    X = (W_total / 3) * 1[the wedge closes into a cross triangle]; unbiased for #cross.
    """
    deg = hg.deg.astype(float)
    W_total = float((deg * (deg - 1) / 2).sum())
    truth = hg.n_cross_total
    if truth <= 0 or W_total <= 0:
        return dict(wedges=W_total, p=0.0, var=np.inf, samples=np.inf, words=np.inf)
    p = 3.0 * truth / W_total
    scale = W_total / 3.0
    var = scale ** 2 * p * (1 - p)
    k = samples_for_error(var, truth, eps)
    return dict(wedges=W_total, p=p, var=var, samples=k, words=k)


def edge_local_cost(hg, eps=EPS):
    """Uniform edge sampling with exact local counting: X = m * t(e) / 3.

    Stronger than our sampler per draw, since it needs a neighbourhood intersection rather
    than a single closing-vertex draw; we report it anyway and say so.
    """
    t = hg.t.astype(float)
    truth = hg.n_cross_total
    x = hg.m * t / 3.0
    var = float(x.var())
    k = samples_for_error(var, truth, eps)
    return dict(var=var, samples=k, words=k * _local_words(hg))


def heavy_storage_cost(hg, taus=None, eps=EPS):
    """Store the top-tau edges by true weight, sample the residual with edge+local counting.

    Total space is tau edge-slots plus the residual sample size; the grid is swept and the
    best operating point reported, which is the setting most favourable to the competitor.
    """
    t = hg.t.astype(float)
    truth = hg.n_cross_total
    order = np.argsort(-t)
    if taus is None:
        taus = np.unique(np.clip(
            np.round(np.geomspace(1, max(hg.m, 2), 25)).astype(int), 1, hg.m))
    best = None
    for tau in taus:
        heavy = order[:tau]
        mask = np.ones(hg.m, dtype=bool)
        mask[heavy] = False
        rest = t[mask]
        m_rest = int(mask.sum())
        if m_rest == 0:
            total = float(tau)
            if best is None or total < best:
                best, best_tau = total, int(tau)
            continue
        # the residual contribution, estimated by uniform sampling over the light edges
        x = m_rest * rest / 3.0
        var = float(x.var())
        k = samples_for_error(var, truth, eps)
        total = float(tau) + k * _local_words(hg, mask)
        if best is None or total < best:
            best, best_tau = total, int(tau)
    return dict(words=best, tau=best_tau if best is not None else None)


# --------------------------------------------------------- simulation of the estimator

def _covbits(hg):
    """Per-edge union of the hyperedges containing both endpoints (as a bitset)."""
    cov = [0] * hg.m
    for i in range(hg.m):
        u, v = int(hg.eu[i]), int(hg.ev[i])
        c = 0
        for h in hg.memb[u] & hg.memb[v]:
            c |= hg.hbits[h]
        cov[i] = c
    return cov


def simulate_estimator(hg, w, k, reps=20, seed=0, cov=None):
    """Run the actual sampler: draw base edge, draw closing vertex, verify, average.

    Returns the array of `reps` estimates of #cross, each from `k` independent instances.
    The estimator is X = 1 / Pr_w[path] on success and 0 otherwise, which is unbiased.
    """
    rng = np.random.default_rng(seed)
    wp = np.asarray(w, dtype=float) + 1.0
    W = wp.sum()
    D = hg._D(wp)
    pe = wp / W
    cum = np.cumsum(wp[hg.slot_eid])
    block_start = np.concatenate([[0.0], cum[hg.indptr[1:-1] - 1]])
    cov = cov if cov is not None else _covbits(hg)
    packed = hg.eu.astype(np.int64) * hg.n + hg.ev.astype(np.int64)
    order = np.argsort(packed)
    packed_sorted = packed[order]

    out = np.empty(reps)
    for r in range(reps):
        base = rng.choice(hg.m, size=k, p=pe)
        u, v = hg.eu[base], hg.ev[base]
        lo = hg.rank[u] < hg.rank[v]
        x = np.where(lo, u, v)
        y = np.where(lo, v, u)
        target = block_start[x] + rng.random(k) * D[x]
        slot = np.searchsorted(cum, target, side='left')
        slot = np.clip(slot, hg.indptr[x], hg.indptr[x + 1] - 1)
        c = hg.indices[slot]
        xz = hg.slot_eid[slot]

        ok = (c != u) & (c != v) & (hg.rank[c] > hg.rank[y])
        # is {y, c} an edge?
        a = np.minimum(y, c).astype(np.int64)
        b = np.maximum(y, c).astype(np.int64)
        key = a * hg.n + b
        pos = np.searchsorted(packed_sorted, key)
        pos = np.clip(pos, 0, len(packed_sorted) - 1)
        ok &= packed_sorted[pos] == key

        idx = np.flatnonzero(ok)
        vals = np.zeros(k)
        for j in idx:                       # the cross test, O(1) per surviving candidate
            if not (cov[base[j]] >> int(c[j])) & 1:
                pr = (wp[base[j]] / W) * (wp[xz[j]] / D[x[j]])
                vals[j] = 1.0 / pr
        out[r] = vals.mean()
    return out


# ------------------------------------------------- the wedge estimator, with predictions

def _third_edge_ids(hg):
    """Edge ids of the third side (y,z) of every cross triangle in the triple arrays.

    The triple arrays give the base edge (x,y) and the closing edge (x,z); the remaining
    side is recovered by a packed-key lookup.  Cached on the CrossHG instance.
    """
    if getattr(hg, '_yz', None) is not None:
        return hg._yz
    b, xz, x = hg.base, hg.xz, hg.pivot
    y = np.where(hg.eu[b] == x, hg.ev[b], hg.eu[b])
    z = np.where(hg.eu[xz] == x, hg.ev[xz], hg.eu[xz])
    a = np.minimum(y, z).astype(np.int64)
    c = np.maximum(y, z).astype(np.int64)
    packed = hg.eu.astype(np.int64) * hg.n + hg.ev.astype(np.int64)
    order = np.argsort(packed)
    ps = packed[order]
    pos = np.searchsorted(ps, a * hg.n + c)
    pos = np.clip(pos, 0, len(ps) - 1)
    hg._yz = order[pos].astype(np.int32)
    return hg._yz


def wedge_weighted_cost(hg, w, eps=EPS):
    """Predictor-weighted uniform-wedge sampling.

    Ordered wedges (x; y, z) are drawn with probability proportional to
    wp(x,y) * wp(x,z), and X = T / (6 * wp(x,y) * wp(x,z)) on a closed cross triangle,
    where T = sum_x (D_x^2 - Q_x) with Q_x = sum_c wp(x,c)^2.  Setting w = 0 recovers
    prediction-free uniform wedge sampling.
    """
    wp = np.asarray(w, dtype=float) + 1.0
    D = hg._D(wp)
    Q = np.add.reduceat(wp[hg.slot_eid] ** 2, hg.indptr[:-1])
    T = float((D ** 2 - Q).sum())
    yz = _third_edge_ids(hg)
    a, b, c = wp[hg.base], wp[hg.xz], wp[yz]
    inv = 1.0 / (a * b) + 1.0 / (a * c) + 1.0 / (b * c)
    m2 = float(inv.sum()) * hg.triple_scale * T / 18.0
    truth = hg.n_cross_total
    var = max(m2 - truth ** 2, 0.0)
    k = samples_for_error(var, truth, eps)
    return dict(T=T, second_moment=m2, var=var, samples=k, words=k)


def wedge_report(hg, n_perm=50, seed=0, eps=EPS):
    """The prediction question for the wedge estimator: does a predictor help, and why?"""
    c_unif = wedge_weighted_cost(hg, hg.w_uniform(), eps)['words']
    c_perf = wedge_weighted_cost(hg, hg.w_perfect(), eps)['words']
    c_mind = wedge_weighted_cost(hg, hg.w_mindeg(), eps)['words']
    perms = np.array([wedge_weighted_cost(hg, hg.w_permuted(hg.w_perfect(), seed + b),
                                          eps)['words'] for b in range(n_perm)])
    s_perf = c_unif / c_perf                      # speedup = cost ratio
    s_perm = c_unif / perms
    gain = s_perf - 1.0
    shares = (s_perf - s_perm) / gain if abs(gain) > 1e-12 else np.full(n_perm, np.nan)
    return dict(
        m=hg.m, n_cross=hg.n_cross_total, copy_density=hg.n_cross_total / max(hg.m, 1),
        wedge_unif_words=c_unif, wedge_perfect_words=c_perf,
        perfect_over_unif=s_perf,
        permuted_over_unif_mean=float(s_perm.mean()),
        permuted_over_unif_lo=float(np.percentile(s_perm, 2.5)),
        permuted_over_unif_hi=float(np.percentile(s_perm, 97.5)),
        mindeg_over_unif=c_unif / c_mind,
        structural_share=float(np.mean(shares)),
        structural_share_lo=float(np.percentile(shares, 2.5)),
        structural_share_hi=float(np.percentile(shares, 97.5)),
        perm_pvalue=float((1 + (s_perm >= s_perf).sum()) / (1 + n_perm)),
    )



def normalizer_inflation(hg):
    """Which normalizer does a skewed weighting inflate?

    The weights are normalised to mean one so that only their skew remains.  W is linear and
    therefore unchanged; D_x is linear but local, and shrinks on average; T is quadratic and
    inflates by convexity.  That difference is why the augmented estimator profits from a
    predictor and the wedge estimator does not.
    """
    wp = hg.w_perfect() + 1.0
    wn = wp / wp.mean()
    one = np.ones(hg.m)

    def T(w):
        D = hg._D(w)
        Q = np.add.reduceat(w[hg.slot_eid] ** 2, hg.indptr[:-1])
        return float((D ** 2 - Q).sum())

    ratio = hg._D(wn) / np.maximum(hg._D(one), 1e-12)
    return dict(
        m=hg.m, weight_cv=float(wn.std()),
        W_inflation=float(wn.sum() / one.sum()),
        D_mean_inflation=float(ratio.mean()), D_max_inflation=float(ratio.max()),
        T_inflation=T(wn) / T(one),
        weighted_gain=weighted_cost(hg, hg.w_uniform())['words'] /
                      weighted_cost(hg, hg.w_perfect())['words'],
        wedge_gain=wedge_weighted_cost(hg, hg.w_uniform())['words'] /
                   wedge_weighted_cost(hg, hg.w_perfect())['words'],
    )


"""The higher-order triadic profile, and its estimation from a stream.

Every triangle of the projection falls into exactly one of three classes:

  covered      some hyperedge contains all three vertices, so the triangle is an artefact
               of the projection rather than evidence of triadic structure;
  transient    a cross-hyperedge triangle at least one of whose pairs co-occurs in only one
               hyperedge -- a triad closed by a one-off encounter;
  persistent   a cross-hyperedge triangle all three of whose pairs recur in two or more
               hyperedges -- a triad closed by repeated relationships.

(Note that a cross triangle always needs three distinct hyperedges: a hyperedge covering two
of its edges would contain all three vertices, which is the covered case.  The classes are
therefore distinguished by pair multiplicity rather than by hyperedge count.)

The profile of a hypergraph is the pair

    f_cross      = #cross / #triangles
    f_persistent = #persistent / #cross

both of which are ratios of quantities the same sampler can estimate.
"""
import numpy as np


__all__ = ['exact_profile', 'simulate_profile', 'domain_separation']


def exact_profile(hg):
    """Exact counts and profile, from the per-edge quantities already computed."""
    n_tri = int((hg.t + hg.covn).sum()) // 3
    n_cross = hg.n_cross_total
    yz = _third_edge_ids(hg)
    mult_min = np.minimum(np.minimum(hg.nhy[hg.base], hg.nhy[hg.xz]), hg.nhy[yz])
    n_pers = float((mult_min >= 2).sum()) * hg.triple_scale
    return dict(
        n_triangles=n_tri, n_cross=n_cross, n_persistent=n_pers,
        f_cross=n_cross / n_tri if n_tri else np.nan,
        f_persistent=n_pers / n_cross if n_cross else np.nan,
    )


def simulate_profile(hg, w, k, reps=20, seed=0, cov=None):
    """Estimate the profile by running the sampler, classifying each discovered triangle.

    One pass of instances yields all three counts at once, since the class is read off the
    triangle the instance already found.
    """
    rng = np.random.default_rng(seed)
    wp = np.asarray(w, dtype=float) + 1.0
    W = wp.sum()
    D = hg._D(wp)
    pe = wp / W
    cum = np.cumsum(wp[hg.slot_eid])
    block_start = np.concatenate([[0.0], cum[hg.indptr[1:-1] - 1]])
    cov = cov if cov is not None else _covbits(hg)
    packed = hg.eu.astype(np.int64) * hg.n + hg.ev.astype(np.int64)
    order = np.argsort(packed)
    ps = packed[order]

    out = np.empty((reps, 3))
    for r in range(reps):
        base = rng.choice(hg.m, size=k, p=pe)
        u, v = hg.eu[base], hg.ev[base]
        lo = hg.rank[u] < hg.rank[v]
        x = np.where(lo, u, v)
        y = np.where(lo, v, u)
        target = block_start[x] + rng.random(k) * D[x]
        slot = np.searchsorted(cum, target, side='left')
        slot = np.clip(slot, hg.indptr[x], hg.indptr[x + 1] - 1)
        c = hg.indices[slot]
        xz = hg.slot_eid[slot]

        ok = (c != u) & (c != v) & (hg.rank[c] > hg.rank[y])
        a = np.minimum(y, c).astype(np.int64)
        b = np.maximum(y, c).astype(np.int64)
        key = a * hg.n + b
        pos = np.clip(np.searchsorted(ps, key), 0, len(ps) - 1)
        hit = ps[pos] == key
        ok &= hit
        yz_id = order[pos]

        tri = np.zeros(k)
        cro = np.zeros(k)
        per = np.zeros(k)
        for j in np.flatnonzero(ok):
            pr = (wp[base[j]] / W) * (wp[xz[j]] / D[x[j]])
            val = 1.0 / pr
            tri[j] = val
            if not (cov[base[j]] >> int(c[j])) & 1:
                cro[j] = val
                mm = min(hg.nhy[base[j]], hg.nhy[xz[j]], hg.nhy[yz_id[j]])
                if mm >= 2:
                    per[j] = val
        out[r] = [tri.mean(), cro.mean(), per.mean()]
    return out


def domain_separation(points, domains, seed=0):
    """Leave-one-out nearest-neighbour accuracy of the domain label in profile space.

    A crude but transparent measure of whether the profile carries domain information; with
    ten graphs in five domains, chance is 1/9 per neighbour under a random pairing.
    """
    P = np.asarray(points, dtype=float)
    P = (P - P.mean(0)) / (P.std(0) + 1e-12)
    dom = np.asarray(domains)
    correct = 0
    for i in range(len(P)):
        d = np.linalg.norm(P - P[i], axis=1)
        d[i] = np.inf
        correct += dom[int(np.argmin(d))] == dom[i]
    return correct / len(P)


"""Ranking vertices by participation in higher-order triadic closure.

For a vertex $v$, let $c(v)$ be the number of cross-hyperedge triangles containing it. This is
obtainable exactly and for free from the per-edge counts, since every such triangle meets $v$
in exactly two $v$-incident edges:

    c(v) = (1/2) * sum_{e incident to v} t(e).

The mining question is whether a stream can identify the top of that ranking without
materialising it, and how much the predictor helps. The evaluation is the standard one:
precision at K against the exact ranking, plus the rank correlation over all vertices.
"""
import numpy as np


__all__ = ['vertex_scores_exact', 'simulate_vertex_scores', 'precision_at_k',
           'rank_correlation']


def vertex_scores_exact(hg):
    """Exact number of cross-hyperedge triangles at each vertex."""
    seg = hg.t.astype(float)[hg.slot_eid]
    return np.add.reduceat(seg, hg.indptr[:-1]) / 2.0


def simulate_vertex_scores(hg, w, k, reps=10, seed=0, cov=None):
    """Estimate the per-vertex scores by running the sampler.

    Each discovered cross triangle contributes 1 / Pr_w[path] to each of its three vertices,
    so one run of k instances produces the whole score vector.
    """
    rng = np.random.default_rng(seed)
    wp = np.asarray(w, dtype=float) + 1.0
    W = wp.sum()
    D = hg._D(wp)
    pe = wp / W
    cum = np.cumsum(wp[hg.slot_eid])
    block_start = np.concatenate([[0.0], cum[hg.indptr[1:-1] - 1]])
    cov = cov if cov is not None else _covbits(hg)
    packed = hg.eu.astype(np.int64) * hg.n + hg.ev.astype(np.int64)
    order = np.argsort(packed)
    ps = packed[order]

    out = np.zeros((reps, hg.n))
    for r in range(reps):
        base = rng.choice(hg.m, size=k, p=pe)
        u, v = hg.eu[base], hg.ev[base]
        lo = hg.rank[u] < hg.rank[v]
        x = np.where(lo, u, v)
        y = np.where(lo, v, u)
        target = block_start[x] + rng.random(k) * D[x]
        slot = np.searchsorted(cum, target, side='left')
        slot = np.clip(slot, hg.indptr[x], hg.indptr[x + 1] - 1)
        c = hg.indices[slot]
        xz = hg.slot_eid[slot]

        ok = (c != u) & (c != v) & (hg.rank[c] > hg.rank[y])
        a = np.minimum(y, c).astype(np.int64)
        b = np.maximum(y, c).astype(np.int64)
        pos = np.clip(np.searchsorted(ps, a * hg.n + b), 0, len(ps) - 1)
        ok &= ps[pos] == a * hg.n + b

        idx = np.flatnonzero(ok)
        keep, vals = [], []
        for j in idx:
            if not (cov[base[j]] >> int(c[j])) & 1:
                pr = (wp[base[j]] / W) * (wp[xz[j]] / D[x[j]])
                keep.append(j)
                vals.append(1.0 / pr)
        if keep:
            keep = np.array(keep)
            vals = np.array(vals) / k
            acc = out[r]
            np.add.at(acc, x[keep], vals)
            np.add.at(acc, y[keep], vals)
            np.add.at(acc, c[keep], vals)
    return out


def precision_at_k(est, exact, K=100):
    """Overlap of the top-K sets of the estimated and exact score vectors."""
    K = min(K, len(exact))
    a = set(np.argsort(-np.asarray(exact))[:K].tolist())
    b = set(np.argsort(-np.asarray(est))[:K].tolist())
    return len(a & b) / K


def rank_correlation(est, exact):
    """Spearman correlation over the vertices that bear at least one copy."""
    from scipy.stats import spearmanr
    exact = np.asarray(exact, dtype=float)
    mask = exact > 0
    if mask.sum() < 3:
        return np.nan
    return float(spearmanr(np.asarray(est)[mask], exact[mask])[0])


In [ ]:
!pip -q install gdown

import gdown, tarfile, glob, os, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 240)
plt.rcParams.update({'font.size': 9, 'figure.dpi': 150, 'savefig.bbox': 'tight'})

ARB_IDS = {
    'contact-primary-school': '1sBHSEIyvVKavAho524Ro4cKL66W6rn-t',
    'contact-high-school':    '1VA2P62awVYgluOIh1W4NZQQgkQCBk-Eu',
    'email-Enron':            '1tTVZkdpgRW47WWmsrdUCukHz0x2M6N77',
    'email-Eu':               '1amLeVudLBDRglCXKlieg6HHE-vu81EVF',
    'NDC-classes':            '1tpDiP1c73O18gCYEx4OI7kx8V_IdYxLt',
    'NDC-substances':         '1mGOg0DMh46J2zQdimSXMde1pKNtfAdh8',
    'DAWN':                   '1wGwoG7oBWnNN7J9TEpjqNpODbsYfMxp4',
    'congress-bills':         '1gH1uJMZpn_SCJSRbORPH4JRQeevLwTyO',
    'tags-math-sx':           '1eDevpF6EZs19rLouNpiKGLIlFOLUfKKG',
    'tags-ask-ubuntu':        '1tb1ZJlXEJnlRkXpTuBZlOqqsFknWCkUV',
}
DOMAIN = {'contact-primary-school':'contact', 'contact-high-school':'contact',
          'email-Enron':'email', 'email-Eu':'email', 'NDC-classes':'drugs',
          'NDC-substances':'drugs', 'DAWN':'drugs', 'congress-bills':'legislation',
          'tags-math-sx':'tags', 'tags-ask-ubuntu':'tags'}

def load_simplices(nv, sp):
    sizes = [int(x) for x in open(nv).read().split()]
    flat = [int(x) for x in open(sp).read().split()]
    out, i = [], 0
    for s in sizes:
        out.append(tuple(flat[i:i + s])); i += s
    return out

def fetch_arb(name, tries=3):
    tgz = f'{name}.tar.gz'
    for k in range(tries):
        try:
            if not os.path.exists(tgz) or os.path.getsize(tgz) < 1000:
                if not gdown.download(id=ARB_IDS[name], output=tgz, quiet=True):
                    gdown.download(url=f'https://drive.google.com/uc?id={ARB_IDS[name]}',
                                   output=tgz, quiet=True, fuzzy=True)
            with tarfile.open(tgz) as t:
                try:
                    t.extractall('.', filter='data')
                except TypeError:
                    t.extractall('.')
            nv = glob.glob(f'**/{name}-nverts.txt', recursive=True)
            sp = glob.glob(f'**/{name}-simplices.txt', recursive=True)
            return load_simplices(nv[0], sp[0])
        except Exception:
            time.sleep(5)
    print(f'  {name}: download failed')
    return None

HG = {}
for name in ARB_IDS:
    h = fetch_arb(name)
    if h is None:
        continue
    uniq = list(dict.fromkeys(tuple(sorted(set(x))) for x in h))
    uniq = [x for x in uniq if 2 <= len(x) <= 25]
    print(f'{name} ...')
    HG[name] = CrossHG(uniq, max_size=25, max_triples=4_000_000, seed=0)

## 1. The exact ranking

The identity $\sum_v c(v) = 3\,\#\cross$ is asserted as a check on the computation. We also
report how concentrated the ranking is, since a flat ranking would make the task trivial and a
degenerate one would make it meaningless.

In [ ]:
rows = []
EXACT = {}
for name, hg in HG.items():
    c = vertex_scores_exact(hg)
    EXACT[name] = c
    assert abs(c.sum() - 3 * hg.n_cross_total) < 1e-6 * max(1, hg.n_cross_total), name
    nz = c[c > 0]
    top = np.sort(c)[::-1]
    rows.append(dict(label=name, domain=DOMAIN[name], n=hg.n, m=hg.m,
                     n_cross=hg.n_cross_total,
                     frac_vertices_with_copies=len(nz) / hg.n,
                     top100_share=top[:100].sum() / c.sum() if c.sum() else np.nan,
                     gini=float((2 * np.arange(1, hg.n + 1) - hg.n - 1).dot(np.sort(c))
                                / (hg.n * c.sum())) if c.sum() else np.nan))
RANKSTATS = pd.DataFrame(rows)
print('identity  sum_v c(v) = 3 #cross  holds on all graphs\n')
RANKSTATS.round(4)

## 2. Recovering the ranking from a stream

Each discovered cross triangle contributes $1/\Pr_{\mathrm w}[\text{path}]$ to each of its
three vertices, so one run of $k$ instances produces the whole score vector. As in the earlier
notebooks the simulation is restricted to graphs below `MAX_M_SIM` edges, because the per-edge
cover bitsets are expensive to build on the largest one.

In [ ]:
MAX_M_SIM = 150_000
K_GRID = [200, 500, 1000, 2000, 5000, 10000, 20000, 50000]
REPS = 8
TOPK = 100

SIM_NAMES = [n for n, h in HG.items() if h.m <= MAX_M_SIM]
print(f'simulating on {len(SIM_NAMES)} of {len(HG)} graphs')
COV = {}
for n in SIM_NAMES:
    t0 = time.time()
    COV[n] = _covbits(HG[n])
    print(f'  cover bitsets for {n}: {time.time() - t0:.1f}s')

In [ ]:
rows = []
for name in SIM_NAMES:
    hg, ex = HG[name], EXACT[name]
    for method, w in [('localized', hg.w_uniform()), ('weighted', hg.w_perfect())]:
        for k in K_GRID:
            est = simulate_vertex_scores(hg, w, k=k, reps=REPS, seed=3, cov=COV[name])
            p = [precision_at_k(e, ex, TOPK) for e in est]
            s = [rank_correlation(e, ex) for e in est]
            rows.append(dict(label=name, domain=DOMAIN[name], method=method, k=k,
                             prec_mean=np.mean(p), prec_std=np.std(p),
                             spearman=np.nanmean(s)))
        r = rows[-1]
        print(f"{name:24s} {method:10s} at k={K_GRID[-1]}: "
              f"P@{TOPK}={r['prec_mean']:.3f}  rho={r['spearman']:.3f}")
RANK = pd.DataFrame(rows)

In [ ]:
agg = RANK.groupby(['method', 'k'])[['prec_mean', 'spearman']].mean().reset_index()
fig, ax = plt.subplots(1, 2, figsize=(8.4, 3.2))
for method, c in [('localized', '#1f77b4'), ('weighted', 'crimson')]:
    d = agg[agg.method == method]
    ax[0].semilogx(d.k, d.prec_mean, 'o-', ms=4, c=c, label=method)
    ax[1].semilogx(d.k, d.spearman, 'o-', ms=4, c=c, label=method)
for a, lab in zip(ax, [f'precision at {TOPK}', "Spearman rank correlation"]):
    a.set_xlabel('instances $k$'); a.set_ylabel(lab)
    a.set_ylim(0, 1.02); a.grid(alpha=.25); a.legend(fontsize=7)
ax[0].axhline(0.8, color='k', lw=.7, ls=':')
fig.savefig('fig8_ranking.pdf'); plt.show()

for target in [0.5, 0.8]:
    for method in ['localized', 'weighted']:
        d = agg[(agg.method == method) & (agg.prec_mean >= target)]
        print(f'P@{TOPK} >= {target}: {method:10s} needs k = '
              f'{int(d.k.min()) if len(d) else "> " + str(K_GRID[-1])}')

In [ ]:
# per-graph view: where does the predictor actually pay?
piv = (RANK[RANK.k == K_GRID[-1]]
       .pivot(index='label', columns='method', values='prec_mean'))
piv['gain'] = piv.weighted - piv.localized
dens = {n: HG[n].n_cross_total / HG[n].m for n in piv.index}
piv['copy_density'] = [dens[n] for n in piv.index]
print(piv.round(3).sort_values('copy_density').to_string())
if len(piv) > 2:
    print(f"\ncorr(gain, copy density) = "
          f"{np.corrcoef(np.log(piv.copy_density), piv.gain)[0, 1]:+.3f}")

## 3. Save and print the LaTeX rows

In [ ]:
RANKSTATS.to_csv('table14_ranking_stats.csv', index=False)
RANK.to_csv('table15_ranking_from_stream.csv', index=False)
print('saved 2 csv files and 1 pdf figure')

print('\n%%%% ---- Table: recovering the vertex ranking from a stream ----')
for _, r in agg.iterrows():
    print(f"{r.method} & {int(r.k)} & {r.prec_mean:.3f} & {r.spearman:.3f} \\\\")

## 4. What this adds to the manuscript

If precision at $100$ reaches a usable level at a few thousand instances, the subsection writes
itself: the primitive supports a per-vertex mining question, the question is answerable from a
stream at a cost far below materialising the exact counts, and the predictor lowers that cost
by the factor the earlier sections predict.

Two things to write honestly. The exact ranking is computed from the same per-edge counts the
perfect predictor uses, so the comparison measures how well the sampler recovers a known
target, not how well a deployed system would do with a learned predictor; the recovery figures
of Section 6.6 bound that separately. And the simulation covers nine of the ten datasets, with
the largest excluded for the same implementation reason as before.

If instead precision stays low at every budget, that is also worth one paragraph: the ranking
is dominated by a heavy tail that sampling cannot resolve cheaply, which would be a limitation
of the primitive rather than of the estimator, and the profile of Notebook 7 remains the
deliverable.